# Practice 3: Customer Clustering - Bước 1: Định nghĩa bài toán (Define Problem)

---

## 1. Bối cảnh doanh nghiệp và Đề bài

Một hãng sản xuất ô tô lớn có kế hoạch thâm nhập vào các thị trường mới bằng các dòng sản phẩm hiện có ($P1, P2, P3, P4, P5,...$). Qua các cuộc khảo sát thị trường sơ bộ, công ty nhận thấy rằng tập khách hàng tiềm năng ở thị trường mới này có những đặc tính và hành vi tương đối đồng nhất với tập khách hàng hiện tại của họ.

Để tối ưu hóa chiến lược tiếp thị (Marketing) và cá nhân hóa trải nghiệm khách hàng, công ty cần chia tập khách hàng thành các phân khúc đặc trưng khác nhau (Customer Segmentation). Nhiệm vụ của chúng ta là phân nhóm (Clustering) khách hàng dựa trên các thuộc tính nhân khẩu học và hành vi tiêu dùng có sẵn trong bộ dữ liệu.

## 3. Định nghĩa toán học của bài toán

### 3.1. Dữ liệu đầu vào (Input $X$)
Gọi tập dữ liệu khách hàng đầu vào là $X = \{x_1, x_2, \dots, x_N\}$, trong đó:
- $N$ là tổng số lượng khách hàng ($N = 8068$ mẫu trong tập `Train.csv`).
- Mỗi khách hàng $x_i$ là một vector đặc trưng $D$ chiều: $x_i = [x_{i1}, x_{i2}, \dots, x_{iD}]^T \in \mathbb{R}^D$ đại diện cho các thuộc tính như `Age`, `Work_Experience`, `Family_Size`, và các thuộc tính phân loại đã được mã hóa (`Gender`, `Ever_Married`, `Graduated`, `Profession`, `Spending_Score`, `Var_1`).

### 3.2. Kết quả đầu ra (Output $C$)
Mục tiêu là tìm một phép gán nhãn phân cụm $C = \{c_1, c_2, \dots, c_N\}$, trong đó:
- $c_i \in \{1, 2, \dots, K\}$ biểu thị chỉ số cụm (Cluster ID) được gán cho khách hàng $x_i$.
- $K$ là số lượng cụm (phân khúc khách hàng) cần xác định (sẽ được tìm kiếm thông qua các độ đo tối ưu).
- Đối với các thuật toán phát hiện nhiễu như DBSCAN, một số điểm có thể được gán nhãn $-1$ (Nhiễu/Noise), không thuộc bất kỳ cụm chính thức nào.

---

## 4. Cơ sở toán học của 4 thuật toán phân cụm

Chúng ta sẽ xây dựng từ đầu (built from scratch) 4 thuật toán phân cụm kinh điển sau:

### 4.1. K-Means Clustering
- **Ý tưởng**: Phân chia dữ liệu thành $K$ cụm sao cho các điểm trong cùng một cụm có khoảng cách đến trọng tâm cụm là nhỏ nhất.
- **Trọng tâm (Centroid)**: $\mu_k = \frac{1}{|S_k|} \sum_{x_i \in S_k} x_i$ (Không bắt buộc là một điểm dữ liệu thực tế).
- **Hàm mục tiêu cần tối thiểu hóa** (Within-Cluster Sum of Squares - WCSS / Inertia):
  $$J_{\text{K-Means}} = \sum_{k=1}^K \sum_{x_i \in S_k} \|x_i - \mu_k\|^2$$
- **Đặc điểm**: Nhạy cảm với outliers do dùng khoảng cách bình phương (L2 norm) và xu hướng tạo ra các cụm hình cầu có kích thước tương đồng.

### 4.2. K-Medoids Clustering
- **Ý tưởng**: Tương tự K-Means nhưng thay vì dùng trung bình số học đại diện làm trọng tâm, K-Medoids chọn một điểm dữ liệu thực tế trong cụm làm đại diện (**Medoid**).
- **Medoid**: Điểm $m_k \in S_k$ thỏa mãn:
  $m_k = \arg\min_{y \in S_k} \sum_{x_i \in S_k} d(x_i, y)$
- **Hàm mục tiêu cần tối thiểu hóa**:
  $J_{\text{K-Medoids}} = \sum_{k=1}^K \sum_{x_i \in S_k} d(x_i, m_k)$
- **Đặc điểm**: Cực kỳ mạnh mẽ với outliers và nhiễu nhờ sử dụng khoảng cách tuyệt đối hoặc ma trận khoảng cách tùy chỉnh (ví dụ: khoảng cách Manhattan).

> **Lưu ý thiết kế trong dự án**: Để đảm bảo tính nhất quán hình học và so sánh công bằng khi trực quan hóa trên không gian giảm chiều PCA (vốn tối ưu hóa phương sai dựa trên khoảng cách Euclidean L2), thuật toán K-Medoids Scratch trong dự án này được đồng nhất sử dụng khoảng cách Euclidean (L2 norm) thay vì Manhattan (L1 norm).

### 4.3. DBSCAN (Density-Based Spatial Clustering of Applications with Noise)
- **Ý tưởng**: Phân cụm dựa trên mật độ điểm dữ liệu. Vùng có mật độ điểm cao tạo thành cụm, vùng thưa thớt bị coi là nhiễu.
- **Tham số cốt lõi**:
  - $\epsilon$ (Epsilon): Bán kính lân cận của một điểm.
  - $MinPts$ (Minimum Points): Số lượng điểm tối thiểu nằm trong bán kính $\epsilon$ (tính cả chính nó).
- **Phân loại điểm**:
  - **Core Point (Điểm lõi)**: Có ít nhất $MinPts$ điểm trong lân cận $\epsilon$.
  - **Border Point (Điểm biên)**: Có ít hơn $MinPts$ điểm trong lân cận $\epsilon$, nhưng nằm trong lân cận $\epsilon$ của một Core Point.
  - **Noise Point (Điểm nhiễu)**: Không phải Core Point cũng không phải Border Point.
- **Đặc điểm**: Tìm được cụm có hình dạng bất kỳ, tự động phát hiện nhiễu và không cần khai báo trước số lượng cụm $K$.

### 4.4. Hierarchical Clustering (Phân cụm phân cấp - Agglomerative)
- **Ý tưởng**: Tiếp cận theo hướng từ dưới lên (Bottom-up). Bắt đầu bằng việc coi mỗi điểm dữ liệu là 1 cụm, sau đó liên tiếp gộp các cụm gần nhau nhất cho đến khi chỉ còn 1 cụm duy nhất (tạo thành cấu trúc cây Dendrogram).
- **Độ đo khoảng cách liên kết (Linkages)**:
  - **Single Linkage** (Gộp dựa trên cặp điểm gần nhất):
    $$d(A, B) = \min \{ d(x, y) : x \in A, y \in B \}$$
  - **Complete Linkage** (Gộp dựa trên cặp điểm xa nhất):
    $$d(A, B) = \max \{ d(x, y) : x \in A, y \in B \}$$
  - **Average Linkage** (Gộp dựa trên khoảng cách trung bình):
    $$d(A, B) = \frac{1}{|A||B|} \sum_{x \in A} \sum_{y \in B} d(x, y)$$
  - **Centroid Linkage** (Gộp dựa trên khoảng cách giữa các trọng tâm):
    $$d(A, B) = d(\mu_A, \mu_B)$$
  - **Ward Linkage** (Gộp dựa trên việc giảm thiểu gia tăng phương sai nội cụm khi sát nhập):
    $$\Delta(A, B) = \frac{|A||B|}{|A|+|B|} \| \mu_A - \mu_B \|^2$$
- **Đặc điểm**: Giúp trực quan hóa cây Dendrogram rất trực quan, dễ dàng phân tích mối quan hệ phân cấp giữa các phân khúc khách hàng.